# 🦴 Osteoporosis Prediction System
## Multi-Model Training Pipeline (XGBoost, SVM, Neural Network)

In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVC
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

---
## 1. LOAD & PREPROCESS DATA

In [2]:
def load_and_preprocess(filepath="osteoporosis.csv"):
    df = pd.read_csv(filepath)
    df.drop(columns=["Id"], inplace=True, errors="ignore")

    for col in ["Alcohol Consumption", "Medical Conditions", "Medications"]:
        for col in df.columns:
            df[col].fillna("None", inplace=True)

    for col in df.select_dtypes(include="object").columns:
        if df[col].isnull().sum() > 0:
            df[col].fillna(df[col].mode()[0], inplace=True)

    bins   = [0, 30, 45, 60, 75, 120]
    labels = ["Young Adult", "Adult", "Middle-Aged", "Senior", "Elderly"]
    df["Age_Group"] = pd.cut(df["Age"], bins=bins, labels=labels, right=False)
    df.drop(columns=["Age"], inplace=True)

    
    encoders = {}
    for col in df.select_dtypes(include=["object", "category"]).columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le  

    
    pickle.dump(encoders, open("encoders.pkl", "wb"))

    X = df.drop(columns=["Osteoporosis"])
    y = df["Osteoporosis"]

    print(f"✅ Dataset loaded: {X.shape[0]} rows, {X.shape[1]} features")
    print(f"   Target classes: {y.value_counts().to_dict()}")

    return X, y

---
## 2. TRAIN-TEST SPLIT + SCALING

In [3]:
def prepare_data(X, y):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, X_train_scaled, X_test_scaled, scaler

---
## 3. MODEL 1 — XGBoost

In [4]:
def train_xgboost(X_train, X_test, y_train, y_test):
    print("\n" + "="*50)
    print("🌲 Training XGBoost...")
    print("="*50)

    model = XGBClassifier(
        n_estimators=300,
        max_depth=4,           
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.6,  
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
        
    )

    model.fit(
        X_train, y_train,
        eval_set=[(X_test, y_test)],
        verbose=False
    )

    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)

    print(f"✅ XGBoost Accuracy: {acc*100:.2f}%")
    print(classification_report(y_test, y_pred))

    return model, acc

---
## 4. MODEL 2 — SVM

In [5]:
def train_svm(X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "="*50)
    print("🔷 Training SVM...")
    print("="*50)

    model = SVC(
        kernel="rbf",
        C=10,
        gamma="scale",
        probability=True,
        random_state=42
    )

    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)

    print(f"✅ SVM Accuracy: {acc*100:.2f}%")
    print(classification_report(y_test, y_pred))

    return model, acc

---
## 5. MODEL 3 — NEURAL NETWORK
**14 Input | 5 Hidden Layers | 1 Output | 100 Epochs**

In [6]:
def build_neural_network(input_dim=14):
    model = Sequential([

        # Hidden Layer 1
        Dense(128, input_dim=input_dim, activation="relu"),  
        BatchNormalization(),
        Dropout(0.3),

        # Hidden Layer 2
        Dense(64, activation="relu"),                        
        BatchNormalization(),
        Dropout(0.3),

        # Hidden Layer 3
        Dense(32, activation="relu"),                         
        BatchNormalization(),
        Dropout(0.2),

        
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    return model

In [7]:
def train_neural_network(X_train_scaled, X_test_scaled, y_train, y_test):
    print("\n" + "="*50)
    print("🧠 Training Neural Network (5 Hidden Layers, 100 Epochs)...")
    print("="*50)

    input_dim = X_train_scaled.shape[1]
    model = build_neural_network(input_dim=input_dim)

    model.summary()

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6)
    ]

    history = model.fit(
        X_train_scaled, y_train,
        epochs=100,
        batch_size=64,
        validation_split=0.15,
        callbacks=callbacks,
        verbose=1
    )

    # Evaluate
    y_pred_prob = model.predict(X_test_scaled)
    y_pred = (y_pred_prob > 0.5).astype(int).flatten()
    acc = accuracy_score(y_test, y_pred)

    print(f"\n✅ Neural Network Accuracy: {acc*100:.2f}%")
    print(classification_report(y_test, y_pred))

    return model, acc, history

---
## 6. AUTO SELECT BEST MODEL

In [8]:
def select_best_model(models_dict):
    print("\n" + "="*50)
    print("🏆 MODEL COMPARISON")
    print("="*50)

    for name, (model, acc) in models_dict.items():
        print(f"  {name:20s} → Accuracy: {acc*100:.2f}%")

    best_name = max(models_dict, key=lambda k: models_dict[k][1])
    best_model, best_acc = models_dict[best_name]

    print(f"\n🥇 BEST MODEL: {best_name} with {best_acc*100:.2f}% accuracy")
    print("="*50)

    return best_name, best_model, best_acc

---
## 7. SAVE MODELS & BEST MODEL INFO

In [9]:
def save_models(xgb_model, svm_model, nn_model, scaler, best_name):
    pickle.dump(xgb_model, open("xgboost_model.pkl", "wb"))
    pickle.dump(svm_model,  open("svm_model.pkl", "wb"))
    pickle.dump(scaler,     open("scaler.pkl", "wb"))
    nn_model.save("neural_network_model.h5")

    # Save best model name
    with open("best_model.txt", "w") as f:
        f.write(best_name)

    print("\n💾 All models saved successfully!")
    print(f"   Best model recorded: {best_name}")

---
## 8. PREDICTION FUNCTION (uses best model)

In [10]:
def predict_osteoporosis(input_data: dict):
    with open("best_model.txt", "r") as f:
        best_name = f.read().strip()

    scaler   = pickle.load(open("scaler.pkl",   "rb"))
    encoders = pickle.load(open("encoders.pkl", "rb"))  

    df_input = pd.DataFrame([input_data])

    # None/NaN values handle karo
    df_input.fillna("None", inplace=True)

    # Age → Age_Group
    if "Age" in df_input.columns:
        bins   = [0, 30, 45, 60, 75, 120]
        labels = ["Young Adult", "Adult", "Middle-Aged", "Senior", "Elderly"]
        df_input["Age_Group"] = pd.cut(df_input["Age"], bins=bins, labels=labels, right=False)
        df_input.drop(columns=["Age"], inplace=True)

    # ✅ CHANGE: Training wale encoders use karo — naya fit mat karo
    for col, le in encoders.items():
        if col in df_input.columns:
            df_input[col] = le.transform(df_input[col].astype(str))

    scaled_input = scaler.transform(df_input)

    print(f"\n🔍 Using Best Model: {best_name}")

    if best_name == "XGBoost":
        model = pickle.load(open("xgboost_model.pkl", "rb"))
        pred = model.predict(df_input)[0]
        prob = model.predict_proba(df_input)[0][1]

    elif best_name == "SVM":
        model = pickle.load(open("svm_model.pkl", "rb"))
        pred = model.predict(scaled_input)[0]
        prob = model.predict_proba(scaled_input)[0][1]

    elif best_name == "Neural Network":
        model = tf.keras.models.load_model("neural_network_model.h5")
        prob = float(model.predict(scaled_input)[0][0])
        pred = int(prob > 0.5)

    result = "Osteoporosis Detected ⚠️" if pred == 1 else "No Osteoporosis ✅"
    print(f"   Prediction : {result}")
    print(f"   Probability: {prob*100:.2f}%")

    return {
        "prediction": int(pred),
        "probability": round(float(prob), 4),
        "result": result,
        "model_used": best_name
    }

---
## 9. MAIN — TRAIN ALL & SELECT BEST

In [11]:
print("\n🦴 OSTEOPOROSIS PREDICTION SYSTEM — STARTING TRAINING\n")

# Step 1: Load Data
X, y = load_and_preprocess("osteoporosis.csv")


🦴 OSTEOPOROSIS PREDICTION SYSTEM — STARTING TRAINING

✅ Dataset loaded: 21958 rows, 14 features
   Target classes: {0: 12952, 1: 9006}


C:\Users\Sahil Gangurde\AppData\Local\Temp\ipykernel_7156\3876071047.py:7: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna("None", inplace=True)


In [12]:
# Step 2: Prepare splits
X_train, X_test, y_train, y_test, \
X_train_scaled, X_test_scaled, scaler = prepare_data(X, y)

In [13]:
# Step 3a: Train XGBoost
xgb_model, xgb_acc = train_xgboost(X_train, X_test, y_train, y_test)


🌲 Training XGBoost...
✅ XGBoost Accuracy: 97.97%
              precision    recall  f1-score   support

           0       0.98      0.99      0.98      2591
           1       0.98      0.97      0.98      1801

    accuracy                           0.98      4392
   macro avg       0.98      0.98      0.98      4392
weighted avg       0.98      0.98      0.98      4392



In [14]:
# Step 3b: Train SVM
svm_model, svm_acc = train_svm(X_train_scaled, X_test_scaled, y_train, y_test)


🔷 Training SVM...
✅ SVM Accuracy: 97.50%
              precision    recall  f1-score   support

           0       0.98      0.98      0.98      2591
           1       0.97      0.97      0.97      1801

    accuracy                           0.97      4392
   macro avg       0.97      0.97      0.97      4392
weighted avg       0.97      0.97      0.97      4392



In [15]:
# Step 3c: Train Neural Network
nn_model, nn_acc, history = train_neural_network(X_train_scaled, X_test_scaled, y_train, y_test)


🧠 Training Neural Network (5 Hidden Layers, 100 Epochs)...


c:\Users\Sahil Gangurde\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 128)            │         1,920 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128)            │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,185 (51.50 KB)

 Trainable params: 12,737 (49.75 KB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.9076 - loss: 0.2582 - val_accuracy: 0.9579 - val_loss: 0.1606 - learning_rate: 0.0010
Epoch 2/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9478 - loss: 0.1744 - val_accuracy: 0.9620 - val_loss: 0.1262 - learning_rate: 0.0010
Epoch 3/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9541 - loss: 0.1514 - val_accuracy: 0.9693 - val_loss: 0.1063 - learning_rate: 0.0010
Epoch 4/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9622 - loss: 0.1317 - val_accuracy: 0.9708 - val_loss: 0.0987 - learning_rate: 0.0010
Epoch 5/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9646 - loss: 0.1179 - val_accuracy: 0.9719 - val_loss: 0.0921 - learning_rate: 0.0010
Epoch 6/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9660 - loss: 0.1135 - val_accuracy: 0.9731 - val_loss: 0.0939 - learning_rate: 0.0010
Epoch 7/100
234/234 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9694 - loss: 0.

In [16]:
# Step 4: Compare & select best
models_dict = {
    "XGBoost":        (xgb_model, xgb_acc),
    "SVM":            (svm_model, svm_acc),
    "Neural Network": (nn_model,  nn_acc),
}

best_name, best_model, best_acc = select_best_model(models_dict)


🏆 MODEL COMPARISON
  XGBoost              → Accuracy: 97.97%
  SVM                  → Accuracy: 97.50%
  Neural Network       → Accuracy: 97.88%

🥇 BEST MODEL: XGBoost with 97.97% accuracy


In [17]:
# Step 5: Save everything
save_models(xgb_model, svm_model, nn_model, scaler, best_name)


💾 All models saved successfully!
   Best model recorded: XGBoost


---
## 🧪 SAMPLE PREDICTION TEST

In [18]:
sample_input = {
    "Age": 50,
    "Gender": "Female",
    "Hormonal Changes": "Postmenopausal",
    "Family History": "Yes",
    "Race/Ethnicity": "African American",
    "Body Weight": "Normal",
    "Calcium Intake": "Low",
    "Vitamin D Intake": "Sufficient",
    "Physical Activity": "Active",
    "Smoking": "No",
    "Alcohol Consumption": "None",
    "Medical Conditions": "None",
    "Medications": "None",
    "Prior Fractures": "No"
}
result = predict_osteoporosis(sample_input)
print(f"\n📊 Final Result: {result}")


🔍 Using Best Model: XGBoost
   Prediction : Osteoporosis Detected ⚠️
   Probability: 96.89%

📊 Final Result: {'prediction': 1, 'probability': 0.9689, 'result': 'Osteoporosis Detected ⚠️', 'model_used': 'XGBoost'}
